# Validation & Evaluation Framework

This notebook establishes and verifies the shared validation and evaluation
framework used across all forecasting models in the project.

The shared evaluation logic is implemented centrally in:

`src/evaluation.py`

This module provides:

- The shared 15-day forecasting horizon
- The documented evaluation-series exclusion rule
- A common evaluation-data filtering function
- A shared evaluation function for consistent metric calculation

The notebook verifies the prepared training and validation split and
documents how all forecasting notebooks should use the shared evaluation
framework.

The models evaluated using this framework include:

- Seasonal Naive
- XGBoost
- LightGBM
- DeepAR

## 1. Objective

The objective of this notebook is to establish a single evaluation framework
for the forecasting stage of the project.

This framework has two main responsibilities:

1. Verify that the training and validation datasets follow a strictly
   chronological split with a 15-day forecasting horizon.

2. Verify and document the shared evaluation logic implemented in
   `src/evaluation.py`, which computes the same performance metrics
   for every forecasting model.

The preprocessing stage has already generated:

- `train_df.parquet`
- `valid_df.parquet`

It validates the existing split and verifies the shared evaluation logic used throughout the modeling stage.

## 2. Imports

The following libraries are used for data inspection and model evaluation.

In [1]:
import sys
import numpy as np
import pandas as pd

sys.path.append("..")

from src.evaluation import (
    FORECAST_HORIZON,
    EXCLUDED_EVALUATION_SERIES,
    filter_evaluation_data,
    evaluate
)

## 3. Project Evaluation Strategy

The project forecasts retail demand at the following granularity:

date × store_nbr × family

After preprocessing and feature engineering, the dataset was divided into
training and validation periods using a strictly chronological split.

The validation period represents the final 15 days of the available
historical data.

The same validation period must be used by every forecasting model:

Training Data
      ↓
Model Training
      ↓
15-Day Validation Forecast
      ↓
Evaluation
      ↓
Model Comparison

This approach ensures that all models are evaluated on the same future
period and under the same conditions.

## 4. Forecast Horizon

The project uses a forecasting horizon of 15 days.

This means that each forecasting model is evaluated by predicting demand
for the same 15-day future period.

The horizon must remain fixed across all models so that performance
comparisons are meaningful.

In [2]:
print("Shared Forecast Horizon:", FORECAST_HORIZON)

Shared Forecast Horizon: 15


## 5. Load Prepared Training and Validation Data

The training and validation datasets were prepared during the preprocessing
and feature engineering stage.

The preprocessing pipeline included:

- Data cleaning
- Structural-zero handling
- Demand pattern classification
- Time-series feature engineering
- Lag and rolling features
- Chronological train-validation splitting

The resulting datasets are loaded directly here to ensure that all
forecasting models use the same prepared data and validation window.

In [6]:
train_df = pd.read_parquet(
    "../data/processed/train_df.parquet"
)

valid_df = pd.read_parquet(
    "../data/processed/valid_df.parquet"
)

## 6. Dataset Overview

Before evaluating the validation strategy, we inspect the prepared datasets
to confirm their structure and time coverage.

In [7]:
print("Training Dataset")
print("Shape:", train_df.shape)

print("\nValidation Dataset")
print("Shape:", valid_df.shape)

Training Dataset
Shape: (2364490, 23)

Validation Dataset
Shape: (25929, 23)


In [8]:
print("\nTraining Date Range:")
print(
    train_df["date"].min().date(),
    "to",
    train_df["date"].max().date()
)

print("\nValidation Date Range:")
print(
    valid_df["date"].min().date(),
    "to",
    valid_df["date"].max().date()
)


Training Date Range:
2013-01-30 to 2017-07-31

Validation Date Range:
2017-08-01 to 2017-08-15


### Insight

The prepared datasets divide the historical data into two consecutive
time periods.

The training data covers:

2013-01-30 → 2017-07-31

The validation data covers:

2017-08-01 → 2017-08-15

This structure allows each model to learn from historical observations
and then generate forecasts for a completely unseen future period.

## 7. Verify Chronological Validation Split

Time-series forecasting requires chronological validation.

Unlike traditional machine learning tasks, observations must not be randomly
shuffled because future information must remain unavailable during training.

We therefore verify that all training observations occur before the
validation period.

In [9]:
train_max_date = train_df["date"].max()

valid_min_date = valid_df["date"].min()

print("Last Training Date:", train_max_date.date())
print("First Validation Date:", valid_min_date.date())

Last Training Date: 2017-07-31
First Validation Date: 2017-08-01


In [10]:
print(
    "\nTraining ends before validation begins:",
    train_max_date < valid_min_date
)


Training ends before validation begins: True


### Insight

The validation period begins after the final training date.

This confirms that the split follows the correct chronological order:

Training Period
      ↓
Future Validation Period

This prevents future sales information from leaking into model training.

## 8. Verify No Date Overlap

We verify that no calendar dates appear in both the training and validation
datasets.

Any overlap would create a risk of data leakage and invalidate the
forecasting evaluation.

In [11]:
train_dates = set(train_df["date"])

valid_dates = set(valid_df["date"])

date_overlap = train_dates.intersection(valid_dates)

print("Overlapping Dates:", date_overlap)

print(
    "Number of Overlapping Dates:",
    len(date_overlap)
)

Overlapping Dates: set()
Number of Overlapping Dates: 0


### Insight

No calendar dates appear in both the training and validation datasets.

This confirms that the validation period is completely held out from
training and can be used as an unseen future period for model evaluation.

## 9. Verify Forecast Horizon

In [12]:
validation_days = valid_df["date"].nunique()

print("Validation Days:", validation_days)

print(
    "Matches Forecast Horizon:",
    validation_days == FORECAST_HORIZON
)

Validation Days: 15
Matches Forecast Horizon: True


### Insight

The validation dataset contains exactly 15 unique calendar days.

This matches the forecasting horizon defined for the project.

Every forecasting model will therefore be evaluated on the same 15-day
future period.

## 10. Verify Store-Family Series Coverage

The project forecasts demand at the following granularity:

date × store_nbr × family

We verify that every Store × Family series present in the validation
data also has training history, since a model cannot be evaluated on
a series it never learned from.

In [15]:
train_series = set(zip(train_df["store_nbr"], train_df["family"]))
valid_series = set(zip(valid_df["store_nbr"], valid_df["family"]))

missing_from_training = valid_series - train_series

print("Training series:", len(train_series))
print("Validation series:", len(valid_series))
print("Series in validation with no training history:", missing_from_training)

Training series: 1728
Validation series: 1729
Series in validation with no training history: {(6, 'BABY CARE')}


### Investigating the cause

Store 6 × BABY CARE has a valid first sale on **2017-07-10** — this is
not a case of the series starting "too late" in absolute terms.

The issue comes from feature engineering: `rolling_mean_28` requires
28 days of prior history before it produces a non-null value. For this
series, that pushes the first row with complete features to
approximately **2017-08-07** — after the training cutoff of
**2017-07-31**. All of its rows are therefore dropped from training by
the `dropna(subset=feature_cols)` step, while a partial window still
remains in the validation period (2017-08-07 to 2017-08-15).

This is an artifact of the rolling-window requirement, not a genuine
data gap — and it does not justify shortening `rolling_mean_28` for the
other 1,728 series, which benefit from the longer window.

The resulting exclusion rule is implemented centrally in
`src/evaluation.py` so that every model applies the same decision.

### Decision

We exclude Store 6 × BABY CARE from model evaluation. A model cannot be
fairly judged on a series it had zero opportunity to learn from — 
scoring it would either penalize every model equally for a data
artifact, or reward whichever model happens to guess closest by chance.

This affects 1 of 1,729 series (0.06%) and does not change the
demand-pattern classification counts, which were computed independently
during preprocessing.

In [17]:
valid_df_eval = filter_evaluation_data(
    valid_df
)
print(
    "Validation rows before exclusion:",
    len(valid_df)
)

print(
    "Validation rows after exclusion:",
    len(valid_df_eval)
)

print(
    "Series excluded:",
    EXCLUDED_EVALUATION_SERIES
)

Validation rows before exclusion: 25929
Validation rows after exclusion: 25920
Series excluded: [(6, 'BABY CARE')]


### Insight

One Store × Family series (Store 6 × BABY CARE) is excluded from
evaluation due to a feature-engineering artifact: the 28-day rolling
window pushes its first valid training row past the training cutoff.

This is documented rather than silently dropped, so every model in the
project — Seasonal Naive, XGBoost, LightGBM, and later DeepAR — uses
the same filtered `valid_df_eval` for scoring, keeping the comparison
consistent across the team.

## 11. Validation Coverage by Demand Pattern

The preprocessing stage classified each Store × Family series into three
demand patterns:

- Regular
- Irregular
- Intermittent

We inspect the distribution of demand patterns in the validation dataset
to ensure that model performance can later be analyzed across different
types of demand behaviour.

In [26]:
evaluation_series = (
    valid_df_eval[
        ["store_nbr", "family", "pattern"]
    ]
    .drop_duplicates()
)

pattern_counts = (
    evaluation_series["pattern"]
    .value_counts()
)

print(pattern_counts)

pattern
Regular         1128
Irregular        378
Intermittent     222
Name: count, dtype: int64


### Insight

After excluding the one documented series with no training rows,
the evaluation dataset contains 1,728 Store × Family series.

The series are distributed as follows:

- 1,128 Regular
- 378 Irregular
- 222 Intermittent

All three demand patterns remain represented in the evaluation dataset.

This allows model performance to be compared both overall and later
across different demand behaviours using the same eligible evaluation
series.

## 12. Evaluation Metrics

Every forecasting model in the project is evaluated using the same four metrics:

- RMSLE
- MAE
- RMSE
- WMAPE


Using multiple metrics provides a broader view of forecasting performance.

RMSLE is the primary metric, while MAE, RMSE, and WMAPE provide
additional perspectives on forecasting performance.

### 12.1 Root Mean Squared Logarithmic Error (RMSLE)

RMSLE is the primary evaluation metric used in this project.

It compares predictions and actual sales values on a logarithmic scale,
which makes it suitable for demand data with:

- Right-skewed sales distributions
- Large differences in sales magnitude
- Many zero-sales observations

Lower RMSLE values indicate better forecasting performance.

### 12.2 Mean Absolute Error (MAE)

MAE measures the average absolute difference between predicted and actual
sales.

It is easy to interpret because it remains in the original sales scale.

Lower MAE values indicate better performance.

### 12.3 Root Mean Squared Error (RMSE)

RMSE measures prediction error while giving larger errors more weight.

This makes RMSE useful for identifying models that make large forecasting
mistakes, particularly during demand spikes.

### 12.4 Weighted Mean Absolute Percentage Error (WMAPE)

WMAPE measures the total absolute forecasting error relative to the
total actual sales.

It expresses forecasting error as a percentage, making it easier to
interpret performance across series with different sales scales.


## 13. Prediction Handling

Sales values cannot be negative.

However, some forecasting models, particularly machine learning models,
may generate negative predictions.

Before calculating any evaluation metric, predictions are clipped to a
minimum value of zero.

For example:

- A prediction of -5 becomes 0
- A prediction of 12 remains 12

This ensures that:

- Forecasts remain valid demand values
- RMSLE can be computed safely
- All models follow the same prediction handling rule
- Evaluation remains consistent across forecasting models

## 14. Shared Evaluation Function

To ensure that every forecasting model uses identical metric calculations,
the evaluation logic is implemented centrally in:

`src/evaluation.py`

The shared function:

`evaluate(actual, prediction, name)`

computes:

- MAE
- RMSE
- RMSLE
- WMAPE

The function also applies the project's shared prediction handling rule by
clipping negative predictions to zero before evaluation.

All forecasting notebooks must import and use this function rather than
redefining metric calculations locally.

In [18]:
print(
    "Shared evaluation function loaded:",
    evaluate
)

Shared evaluation function loaded: <function evaluate at 0x0000014E5993D4E0>


## 15. Test the Evaluation Function

Before using the evaluation function with forecasting models, we test it
using a small example.

This confirms that the function returns all required metrics and applies
the shared prediction handling rule correctly.

In [19]:
actual_example = [10, 20, 30, 40]

prediction_example = [12, 18, 35, 38]

example_results = evaluate(
    actual_example,
    prediction_example,
    name="Example"
)

example_results

Example              MAE:     2.75 | RMSE:     3.04 | RMSLE: 0.1253 | WMAPE:  11.00%


{'Model': 'Example',
 'MAE': 2.75,
 'RMSE': np.float64(3.0413812651491097),
 'RMSLE': np.float64(0.1252842243758099),
 'WMAPE': np.float64(11.0)}

### Insight

The test confirms that the shared evaluation function returns all four
required metrics from the same input interface.

It also confirms that the same prediction handling rule is applied before
all metrics are calculated.

This allows every forecasting model to be evaluated using identical metric
calculations and evaluation rules.

## 16. Handoff to the Modeling Stage

The shared validation and evaluation framework is now ready for use by all
forecasting notebooks.

The following steps describe how each model notebook should integrate the
shared evaluation logic.

These are implementation guidelines for the modeling notebooks.
Model-specific objects such as trained models, feature matrices, and
prediction variables are created inside each model's own notebook.

### Step 1. Import the shared evaluation utilities

Each forecasting notebook should import the shared evaluation module instead
of redefining the evaluation metrics or exclusion logic locally.

This ensures that all models use the same:

- Forecast horizon
- Evaluation-series exclusion rule
- Evaluation-data filtering
- Metric calculations

In [24]:
import sys

sys.path.append("..")

from src.evaluation import (
    FORECAST_HORIZON,
    EXCLUDED_EVALUATION_SERIES,
    filter_evaluation_data,
    evaluate
)

### Step 2. Apply the shared evaluation filter

Before evaluating model predictions, the validation dataset must be filtered
using the shared function.

This removes the documented Store 6 × BABY CARE series consistently across
all models.

In [25]:
valid_df_eval = filter_evaluation_data(
    valid_df
)

### Step 3. Generate model predictions

Each forecasting notebook is responsible for generating its own predictions.

The prediction process depends on the model implementation.

For example:

- Seasonal Naive uses historical seasonal values
- XGBoost generates predictions from its trained XGBoost model
- LightGBM generates predictions from its trained LightGBM model
- DeepAR generates predictions from its trained probabilistic forecasting model

The resulting predictions must correspond to the same rows in
`valid_df_eval`.

### Step 4. Evaluate predictions using the shared function

After predictions are generated, each model must use the shared
`evaluate()` function.

The actual sales and predictions must refer to the same filtered validation
rows.

The function automatically applies the project's prediction handling rule
and calculates RMSLE, MAE, RMSE, and WMAPE.

The following example should be implemented inside the corresponding
modeling notebook after its prediction variable has been created.

```python
results = evaluate(
    actual=valid_df_eval["sales"],
    prediction=predictions,
    name="Model Name"
)


### Step 5. Save detailed predictions for downstream analysis

Detailed predictions should be preserved so that forecasting performance can
later be analyzed by demand pattern and compared across models.

Each model's prediction results should remain aligned with the filtered
evaluation dataset.

For cross-model analysis, completed model predictions should be combined
into a shared dataset containing:

- date
- store_nbr
- family
- pattern
- sales
- one prediction column for each model

The combined dataset is used by the Demand Pattern Evaluation notebook.

```python 
date
store_nbr
family
pattern
sales
seasonal_naive_pred
xgboost_pred
lightgbm_pred
deepar_pred

# Final Summary

This notebook establishes and verifies the shared validation and evaluation
framework used throughout the forecasting stage of the project.

The framework confirms that:

- The data follows a strictly chronological split
- The validation period contains exactly 15 forecast days
- Training and validation dates do not overlap
- The same validation window is used across all forecasting models
- One documented Store × Family series is excluded consistently from evaluation
- RMSLE is the primary evaluation metric
- MAE, RMSE, and WMAPE are supporting metrics
- Negative predictions are clipped to zero before evaluation
- Shared evaluation logic is centralized in `src/evaluation.py`
- All modeling notebooks use the same evaluation filtering and metric calculations
- Detailed predictions can be preserved for downstream demand-pattern analysis

This framework ensures that Seasonal Naive, XGBoost, LightGBM, and DeepAR
are evaluated under consistent conditions, allowing their forecasting
performance to be compared fairly across the same validation period.